# Problem 026:  Reciprocal Cycles

> A unit fraction contains $1$ in the numerator. The decimal representation of the unit fractions with denominators $2$ to $10$ are given:
>
> $$\begin{align}
1/2 &= 0.5\\
1/3 &=0.(3)\\
1/4 &=0.25\\
1/5 &= 0.2\\
1/6 &= 0.1(6)\\
1/7 &= 0.(142857)\\
1/8 &= 0.125\\
1/9 &= 0.(1)\\
1/10 &= 0.1
\end{align}$$
>
> Where $0.1(6)$ means $0.166666\cdots$, and has a $1$-digit recurring cycle. It can be seen that $1/7$ has a $6$-digit recurring cycle.
>
> Find the value of $d \lt 1000$ for which $1/d$ contains the longest recurring cycle in its decimal fraction part.

## Naive Solution $\mathcal O(N^2)$

Let's divide the problem into two parts.  The first part is writing an algorithm to find the length of the recurring cycle for $1/n$.  The second is to find which value of $n < N$ maximizes the cycle.

At first, it would seem that solving for the decimal expansion of $1/n$ and finding when it repeats would be the way to solve the problem.  Whereas this does work, there are a few issues with this approach.  One is that there can be transient values at the start of the decimal expansion, as seen above for the example of $1/6$ that starts with a $1$ before the repeating $6$.  Identifying the repeating pattern becomes a little cumbersome if you need to check that it's repeating starting at any previous value.

Instead of looking at the digits produced in the decimal expansion, consider the remainder produced in each step.  For clarification of what I mean, consider the method that would be used to find the decimal expansion, $d_k$, of $1/n$ by hand considering the remainder $r_k$ after each step:
$$d_0 = 0\quad \mbox{and} \quad r_0 = 1$$
$$d_k = \left\lfloor \frac{10\, r_{k-1}}{n} \right\rfloor \quad \mbox{and} \quad r_k = 10\, r_{k-1} \mod n\quad \mbox{for} \quad k>0$$
In this sense, it is not only the values of $d_k$ that are repeating, but also $r_k$.  Working with $r_k$ also avoids working with $d_k$ at all, since the recurrence equation for $r_k$ only involves the previous $r_{k-1}$.

The algorithm then consists of making an array `seen` of size $n$ initialized to $0$ and changing the value at index $r_k$ each step with the step number $k$.  As soon as the remainder $r_k$ is seen a second time, the pattern has been reached and the size of the recurrence is the difference in the current step number and the step number stored in `seen`.  The code also identifies when the decimal expansion terminates, and returns a recurrence of $0$ for that situation.

As for the second part, since the first part is at worst a $\mathcal O(n)$ process, why not just calculate it for all values less than $N$ and just return the $n$ of the maximum repetend found.  This would make it an $\mathcal O(N^2)$ process overall.

In [19]:
%%time

def repetend(x: int) -> int:
    remainder = 10 % x
    seen = [0] * x
    n = 1
    while seen[remainder] == 0:
        seen[remainder] = n
        remainder = (remainder * 10) % x
        n += 1
    if not remainder:
        return 0
    return n - seen[remainder]

def p026_naive(N: int) -> int:
    return max([(repetend(i),i) for i in range(2,N+1)])[1]

N = 1000
ans = p026_naive(N)

print(ans)

983
CPU times: user 18.9 ms, sys: 1.83 ms, total: 20.7 ms
Wall time: 20.2 ms


## Solution $\mathcal O(N\ln N)$

Not all values below $N$ need to be calculated.  Observe that the longest repetend of $n$ can be no larger than $n-1$.  Thus starting at the largest value of $n$ and decrementing at each step allows for early termination of the code as soon as the largest repetend seen is greater than $n$.

As for time scaling, a more concise statement of when the repetend of $n$ is of size $n-1$ is found in the Wikipedia page for [Repeating decimal](https://en.wikipedia.org/wiki/Repeating_decimal):
> For an arbitrary integer $n$, the length $L(n)$ of the decimal repetend of $⁠1/n$⁠ divides $\phi(n)$, where $\phi$ is the totient function. The length is equal to $\phi(n)$ if and only if $10$ is a primitive root modulo $n$.

I will save discussions of the totient function and primitive roots modulo $n$ for later problems, but suffice it to say that $\phi(n) = n - 1$ for prime $n$.  Thus the search depth of the algorithm will be proportional to the distance between prime numbers around $N$, which is $\mathcal O(\ln N)$.

In [26]:
%%time

def repetend(x: int) -> int:
    remainder = 10 % x
    seen = [0] * x
    n = 1
    while seen[remainder] == 0:
        seen[remainder] = n
        remainder = (remainder * 10) % x
        n += 1
    if not remainder:
        return 0
    return n - seen[remainder]

def p026(N: int) -> int:
    val = (0, 0)
    n = N - 1
    while val[0] < n:
        val = max(val, (repetend(n), n))
        n -= 1
    return val[1]

N = 1_000
ans = p026(N)

print(ans)

983
CPU times: user 878 μs, sys: 0 ns, total: 878 μs
Wall time: 865 μs


The above statement of totient functions and primitive roots modulo $n$ may suggest a more efficient way of finding the answer, but not in any useful way that I can think of.  In fact, the longest repetend up to a value $M$ is not always a prime, as seen in the code below.  The overhead of finding the potential values $n$ with long repetends and weeding those that are small would outweigh the simplicity of using the `repetend` function presented above.

In [33]:
%%time

val = 0
for i in range(2,10_000):
    x = repetend(i)
    if x > val:
        if x != i-1:
            print(i,x)
        val = x

3 1
289 272
361 342
CPU times: user 1.92 s, sys: 977 μs, total: 1.92 s
Wall time: 1.95 s
